# Validation automatique des mesures morphométriquesPour chaque mesure (27), on classe l'image en **mesurable / non mesurable** à partir desconfiances de keypoints prédites par le modèle de pose.**Convention : la classe positive est `non mesurable`** (classe minoritaire, celle qui nous intéresse).**8 approches comparées :**| clé | features ||---|---|| `seuil_conf_moy` | moyenne des confiances des kp directs de la mesure || `seuil_conf_min` | minimum des confiances des kp directs de la mesure || `xgb_direct` / `rf_direct` | confiances des kp directs + one-hot groupe || `xgb_related` / `rf_related` | confiances du voisinage anatomique (`related_entities`) + one-hot groupe || `xgb_all` / `rf_all` | confiances des 42 kp + one-hot groupe |**Protocole :** 5-fold stratifié, prédictions out-of-fold. Le seuil de décision est choisisur une partition interne du *train* (maximisation du MCC), jamais sur le test.**Métrique de comparaison inter-mesures : le MCC** (robuste au déséquilibre), complété parl'AP normalisée `(AP - prévalence) / (1 - prévalence)` qui corrige l'effet de prévalence.

## 1. Configuration

In [ ]:
from __future__ import annotations

import gc
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_curve,
)
from xgboost import XGBClassifier

from insect_anatomy import INSECT_GROUPS, MEAS_TO_KP, MEASUREMENTS, POINTS, related_entities
from dataset import STATUS_SUFFIX, build_dataset, label_report

warnings.filterwarnings("ignore")
matplotlib.use("Agg")  # pas de fenêtre : on écrit des PNG

# --- chemins (à adapter) ---------------------------------------------------
DATA_DIR = Path("data")                       # CSV d'annotation
DATABASE_DIR = Path("database")               # arborescence images/<groupe>/
RESULTS_PATH = Path("results/pose_results.csv")  # sorties du modèle de pose
OUT_DIR = Path("outputs")

# --- protocole -------------------------------------------------------------
N_FOLDS = 5
RANDOM_STATE = 0
MIN_MINORITY = 20   # garde-fou : mesure ignorée en dessous de ce nombre
NA_FILL = -1.0      # sentinelle d'imputation pour la random forest

for sub in ("pr_curves", "confusion"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)


def slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

## 2. Chargement des données

In [ ]:
frame, columns = build_dataset(DATA_DIR, DATABASE_DIR, RESULTS_PATH)
GROUP_COLS = [f"{g}_one_hot" for g in INSECT_GROUPS]

print(f"{len(frame)} images | {columns.summary()}")
frame[GROUP_COLS].sum().rename("n images").to_frame().T

In [ ]:
# Prévalence par mesure : sert au garde-fou et à la lecture des accuracies.
prevalence = []
for measure in MEASUREMENTS:
    status = f"{measure}{STATUS_SUFFIX}"
    if status not in frame.columns:
        continue
    y = 1 - frame[status].astype(int).to_numpy()   # 1 = non mesurable
    prevalence.append({
        "measure": measure,
        "n": len(y),
        "n_unmeasurable": int(y.sum()),
        "prevalence_unmeasurable": float(y.mean()),
        "kept": bool(min(y.sum(), len(y) - y.sum()) >= MIN_MINORITY),
    })

prevalence = pd.DataFrame(prevalence)
prevalence.to_csv(OUT_DIR / "prevalence.csv", index=False)

KEPT = prevalence.loc[prevalence["kept"], "measure"].tolist()
SKIPPED = prevalence.loc[~prevalence["kept"], "measure"].tolist()
print(f"{len(KEPT)} mesures retenues, {len(SKIPPED)} ignorées (< {MIN_MINORITY} exemples minoritaires)")
print("ignorées :", SKIPPED)
prevalence

## 3. Jeux de features et approches

In [ ]:
def conf_columns(points) -> list:
    """Colonnes de confiance existantes pour une liste de keypoints."""
    return [columns.conf[p] for p in points if p in columns.conf]


ALL_CONF = conf_columns(POINTS)


def feature_sets(measure: str) -> dict:
    return {
        "direct": conf_columns(MEAS_TO_KP[measure]),
        "related": conf_columns(related_entities(measure)[0]),
        "all": ALL_CONF,
    }


def target(measure: str) -> np.ndarray:
    """1 = non mesurable (classe positive, minoritaire)."""
    return 1 - frame[f"{measure}{STATUS_SUFFIX}"].astype(int).to_numpy()


def make_xgb(y_train: np.ndarray) -> XGBClassifier:
    n_pos = max(int(y_train.sum()), 1)
    n_neg = max(len(y_train) - n_pos, 1)
    return XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9,
        scale_pos_weight=n_neg / n_pos,          # déséquilibre
        eval_metric="logloss", tree_method="hist",
        n_jobs=-1, random_state=RANDOM_STATE,
    )


def make_rf(y_train: np.ndarray) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=200, min_samples_leaf=2,
        class_weight="balanced",                  # déséquilibre
        n_jobs=-1, random_state=RANDOM_STATE,
    )


# (clé, type, jeu de features, fabrique de modèle)
APPROACHES = [
    ("seuil_conf_moy", "rule", "direct", None),
    ("seuil_conf_min", "rule", "direct", None),
    ("xgb_direct", "model", "direct", make_xgb),
    ("rf_direct", "model", "direct", make_rf),
    ("xgb_related", "model", "related", make_xgb),
    ("rf_related", "model", "related", make_rf),
    ("xgb_all", "model", "all", make_xgb),
    ("rf_all", "model", "all", make_rf),
]
NAMES = [a[0] for a in APPROACHES]

## 4. Score, seuil et validation croiséeToutes les approches produisent un **score croissant avec la probabilité de « non mesurable »**,ce qui rend courbes PR et matrices de confusion directement comparables.Un keypoint non détecté (cellule vide) vaut une confiance nulle pour les règles de seuil,reste `NaN` pour XGBoost (branche informative native) et est imputé à `-1` pour la random forest.

In [ ]:
def rule_score(values: pd.DataFrame, how: str) -> np.ndarray:
    data = np.nan_to_num(values.to_numpy(dtype=float), nan=0.0)  # kp manquant -> conf 0
    agg = data.mean(axis=1) if how == "mean" else data.min(axis=1)
    return -agg   # confiance basse -> score élevé -> non mesurable


def best_threshold(y: np.ndarray, score: np.ndarray) -> float:
    """Seuil maximisant le MCC, cherché sur des données non vues à l'entraînement."""
    candidates = np.unique(np.quantile(score, np.linspace(0.0, 1.0, 201)))
    best_t, best_m = candidates[0], -np.inf
    for t in candidates:
        m = matthews_corrcoef(y, (score >= t).astype(int))
        if m > best_m:
            best_t, best_m = t, m
    return float(best_t)


def evaluate_measure(measure: str) -> dict:
    """Prédictions out-of-fold des 8 approches pour une mesure."""
    y = target(measure)
    sets = feature_sets(measure)
    splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    scores = {name: np.zeros(len(y)) for name in NAMES}
    preds = {name: np.zeros(len(y), dtype=int) for name in NAMES}
    thresholds = {name: [] for name in NAMES}

    for train_idx, test_idx in splitter.split(np.zeros(len(y)), y):
        # partition interne : sert uniquement à fixer le seuil de décision
        inner_fit, inner_val = train_test_split(
            train_idx, test_size=0.25, stratify=y[train_idx], random_state=RANDOM_STATE
        )
        for name, kind, which, factory in APPROACHES:
            cols = sets[which]
            if not cols:
                continue
            if kind == "rule":
                how = "mean" if name.endswith("moy") else "min"
                val_score = rule_score(frame.iloc[inner_val][cols], how)
                test_score = rule_score(frame.iloc[test_idx][cols], how)
            else:
                data = frame[cols + GROUP_COLS]
                if name.startswith("rf"):
                    data = data.fillna(NA_FILL)   # la RF ne gère pas les NaN
                model = factory(y[inner_fit])
                model.fit(data.iloc[inner_fit], y[inner_fit])
                val_score = model.predict_proba(data.iloc[inner_val])[:, 1]
                del model
                gc.collect()
                model = factory(y[train_idx])
                model.fit(data.iloc[train_idx], y[train_idx])
                test_score = model.predict_proba(data.iloc[test_idx])[:, 1]
                del model
                gc.collect()

            threshold = best_threshold(y[inner_val], val_score)
            scores[name][test_idx] = test_score
            preds[name][test_idx] = (test_score >= threshold).astype(int)
            thresholds[name].append(threshold)

    return {"y": y, "scores": scores, "preds": preds, "thresholds": thresholds}

## 5. Métriques et figures

In [ ]:
def metric_rows(measure: str, result: dict) -> list:
    y = result["y"]
    prev = float(y.mean())
    rows = []
    for name in NAMES:
        score, pred = result["scores"][name], result["preds"][name]
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        ap = float(average_precision_score(y, score))
        rows.append({
            "measure": measure,
            "model": name,
            "n": len(y),
            "prevalence_unmeasurable": prev,
            "mcc": float(matthews_corrcoef(y, pred)),
            "average_precision": ap,
            "average_precision_norm": (ap - prev) / (1 - prev) if prev < 1 else np.nan,
            "accuracy": float((tp + tn) / len(y)),
            "accuracy_unmeasurable": float(tp / (tp + fn)) if (tp + fn) else np.nan,
            "accuracy_measurable": float(tn / (tn + fp)) if (tn + fp) else np.nan,
            "balanced_accuracy": float(0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1))),
            "precision_unmeasurable": float(tp / (tp + fp)) if (tp + fp) else np.nan,
            "threshold_median": float(np.median(result["thresholds"][name])),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        })
    return rows


def plot_pr(measure: str, result: dict) -> None:
    y = result["y"]
    fig, ax = plt.subplots(figsize=(7, 5.5))
    for name in NAMES:
        score = result["scores"][name]
        precision, recall, _ = precision_recall_curve(y, score)
        ap = average_precision_score(y, score)
        ax.plot(recall, precision, lw=1.6, label=f"{name} (AP={ap:.3f})")
    ax.axhline(y.mean(), color="grey", ls="--", lw=1, label=f"hasard ({y.mean():.3f})")
    ax.set_xlabel("Rappel (non mesurable)")
    ax.set_ylabel("Précision (non mesurable)")
    ax.set_title(f"Courbe précision-rappel — {measure}")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7, loc="lower left")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "pr_curves" / f"{slug(measure)}.png", dpi=140)
    plt.close(fig)


def plot_confusion(measure: str, result: dict) -> None:
    y = result["y"]
    labels = ["mesurable", "non mes."]
    fig, axes = plt.subplots(2, 4, figsize=(13, 7))
    for ax, name in zip(axes.ravel(), NAMES):
        matrix = confusion_matrix(y, result["preds"][name], labels=[0, 1])
        normalised = matrix / matrix.sum(axis=1, keepdims=True).clip(min=1)
        ax.imshow(normalised, cmap="Blues", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{matrix[i, j]}\n{normalised[i, j]:.0%}",
                        ha="center", va="center", fontsize=9,
                        color="white" if normalised[i, j] > 0.5 else "black")
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_yticklabels(labels, fontsize=8)
        ax.set_title(f"{name}\nMCC={matthews_corrcoef(y, result['preds'][name]):.3f}", fontsize=9)
        ax.set_xlabel("prédit", fontsize=8)
        ax.set_ylabel("réel", fontsize=8)
    fig.suptitle(f"Matrices de confusion (out-of-fold) — {measure}")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "confusion" / f"{slug(measure)}.png", dpi=140)
    plt.close(fig)

## 6. Boucle principale

In [ ]:
rows = []
for i, measure in enumerate(KEPT, start=1):
    print(f"[{i:2d}/{len(KEPT)}] {measure}", flush=True)
    result = evaluate_measure(measure)
    rows.extend(metric_rows(measure, result))
    plot_pr(measure, result)
    plot_confusion(measure, result)
    del result           # libère les scores avant la mesure suivante
    gc.collect()

metrics = pd.DataFrame(rows)
metrics.to_csv(OUT_DIR / "metrics.csv", index=False)
print(f"\n{len(metrics)} lignes écrites dans {OUT_DIR / 'metrics.csv'}")
metrics.head(8)

## 7. Comparaison des approchesClassement par **rang moyen du MCC** sur l'ensemble des mesures retenues(rang 1 = meilleure approche pour la mesure), plus le nombre de victoires.

In [ ]:
mcc = metrics.pivot(index="measure", columns="model", values="mcc")[NAMES]
ranks = mcc.rank(axis=1, ascending=False)

ranking = pd.DataFrame({
    "mean_rank": ranks.mean(),
    "median_mcc": mcc.median(),
    "mean_mcc": mcc.mean(),
    "mean_ap_norm": metrics.pivot(index="measure", columns="model",
                                  values="average_precision_norm")[NAMES].mean(),
    "mean_accuracy_unmeasurable": metrics.pivot(index="measure", columns="model",
                                                values="accuracy_unmeasurable")[NAMES].mean(),
    "wins": mcc.idxmax(axis=1).value_counts().reindex(NAMES).fillna(0).astype(int),
}).sort_values("mean_rank")

ranking.to_csv(OUT_DIR / "ranking.csv")
ranking

In [ ]:
# Heatmap mesure x approche (MCC)
fig, ax = plt.subplots(figsize=(9, 0.42 * len(mcc) + 2.5))
image = ax.imshow(mcc.to_numpy(), cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(NAMES)))
ax.set_yticks(range(len(mcc)))
ax.set_xticklabels(NAMES, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(mcc.index, fontsize=8)
for i in range(mcc.shape[0]):
    for j in range(mcc.shape[1]):
        value = mcc.iat[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=6.5,
                color="white" if value < 0.6 else "black")
fig.colorbar(image, ax=ax, label="MCC")
ax.set_title("MCC out-of-fold par mesure et par approche")
fig.tight_layout()
fig.savefig(OUT_DIR / "heatmap_mcc.png", dpi=140)
plt.close(fig)

# Barplot du rang moyen
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(ranking.index[::-1], ranking["mean_rank"][::-1], color="steelblue")
for name, value in zip(ranking.index[::-1], ranking["mean_rank"][::-1]):
    ax.text(value + 0.05, name, f"{value:.2f}", va="center", fontsize=8)
ax.set_xlabel(f"Rang moyen du MCC sur {len(mcc)} mesures (1 = meilleur)")
ax.set_title("Classement des approches")
fig.tight_layout()
fig.savefig(OUT_DIR / "ranking.png", dpi=140)
plt.close(fig)

print("Figures écrites :", len(list(OUT_DIR.rglob('*.png'))))

## 8. Lecture des résultats- `outputs/metrics.csv` — toutes les métriques, une ligne par (mesure, approche).- `outputs/prevalence.csv` — effectifs et mesures écartées par le garde-fou.- `outputs/ranking.csv` + `ranking.png` — classement global des 8 approches.- `outputs/heatmap_mcc.png` — quelles mesures sont difficiles, indépendamment de l'approche.- `outputs/pr_curves/*.png`, `outputs/confusion/*.png` — détail par mesure.Deux points de vigilance à l'interprétation :1. `accuracy` doit toujours être lue face à `1 - prevalence_unmeasurable` (accuracy du   classifieur constant). Une accuracy de 0,95 sur une mesure à 5 % de non-mesurables ne vaut rien.2. Les mesures d'ailes postérieures sont labellisées non mesurables *par construction* chez les   groupes qui n'en ont pas : le MCC y sera très élevé simplement parce que le one-hot de groupe   suffit. Le tableau ci-dessous isole ces cas.

In [ ]:
# Mesures dont le label est quasi déterminé par le groupe taxonomique.
suspects = []
for measure in KEPT:
    y = pd.Series(target(measure))
    by_group = y.groupby(frame["group"].astype(str).to_numpy()).mean()
    if ((by_group < 0.05) | (by_group > 0.95)).all():
        suspects.append({"measure": measure, **by_group.round(2).to_dict()})

suspects = pd.DataFrame(suspects)
if not suspects.empty:
    suspects.to_csv(OUT_DIR / "group_determined_measures.csv", index=False)
    print("Taux de non-mesurable par groupe (mesures triviales) :")
suspects